In [12]:
import os, re, csv
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = "c:/Users/lava0/Desktop/한국외대/사회과학과 데이터 사이언스"

PAIRS = f"{DATA_DIR}/MicroLens-100k_pairs.csv"
LIKES = f"{DATA_DIR}/MicroLens-100k_likes_and_views.txt"
COMMENT = f"{DATA_DIR}/MicroLens-100k_comment_en.txt"
TITLE = f"{DATA_DIR}/MicroLens-100k_title_en.csv"

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

from matplotlib import font_manager
_installed = {f.name for f in font_manager.fontManager.ttflist}

for _cand in ["NanumGothic", "NanumSquareRound", "NanumBarunGothic", "Noto Sans CJK KR",
"Noto Sans CJK SC", "Malgun Gothic", "AppleGothic"]:
    if _cand in _installed:
        plt.rcParams["font.family"] = _cand
        print("plot font:", _cand)
        break
else:
    print("한글 폰트 찾지 못함")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

for p in [PAIRS, LIKES, COMMENT, TITLE]:
    print(f"{os.path.basename(p):45s} {os.path.getsize(p)/1e6:8.1f} MB")

plot font: NanumGothic
MicroLens-100k_pairs.csv                          18.9 MB
MicroLens-100k_likes_and_views.txt                 0.4 MB
MicroLens-100k_comment_en.txt                     38.2 MB
MicroLens-100k_title_en.csv                        1.5 MB


In [13]:
pairs_raw = pd.read_csv(PAIRS)
likes_raw = pd.read_csv(LIKES, sep="\t", header=None, names=["items", "likes", "views" ])
title_raw = pd.read_csv(TITLE, header=None, names=["items", "title"])
title_raw.head()

,items,title
0,1,Gu long song gaga bad
1,2,beautiful dish with 5 pounds of duck intestines
2,3,This sudden reversal
3,4,pumpkin play armor new outfit
4,5,the wife a day brush 200


In [21]:
def quality_report(name, df, key_cols):
    na = df.isna().sum()
    rep = pd.DataFrame({"결측치" : na, "결측치 비율(%)" : (na/len(df)*100).round(3)})
    print(f"\n === {name} ({len(df) :,} 행) ===")
    display(rep)
    print(f"완전 중복 행         : {df.duplicated().sum():,}")
    print(f"키 {key_cols} 중복 행 : {df.duplicated(subset=key_cols).sum():,}")


quality_report("PAIRS", pairs_raw, ["user", "item"])
quality_report("LIKES", likes_raw, ["items"])
quality_report("TITLE", title_raw, ["items"])


 === PAIRS (719,405 행) ===


,결측치,결측치 비율(%)
user,0,0.0
item,0,0.0
timestamp,0,0.0


완전 중복 행         : 0
키 ['user', 'item'] 중복 행 : 0

 === LIKES (19,738 행) ===


,결측치,결측치 비율(%)
items,0,0.0
likes,0,0.0
views,0,0.0


완전 중복 행         : 0
키 ['items'] 중복 행 : 0

 === TITLE (19,738 행) ===


,결측치,결측치 비율(%)
items,0,0.0
title,0,0.0


완전 중복 행         : 0
키 ['items'] 중복 행 : 0


In [16]:
pairs = pairs_raw.copy()
pairs["datetime"] = pd.to_datetime(pairs["timestamp"], unit="ms")
print(pairs.dtypes)
pairs.head()

user                  int64
item                  int64
timestamp             int64
datetime     datetime64[ms]
dtype: object


,user,item,timestamp,datetime
0,36121,9580,1583378629552,2020-03-05 03:23:49.552
1,26572,9580,1583436719018,2020-03-05 19:31:59.018
2,94805,9580,1584083806481,2020-03-13 07:16:46.481
3,37550,9580,1584412681021,2020-03-17 02:38:01.021
4,89825,9580,1584649439020,2020-03-19 20:23:59.020


In [17]:
def percentile_level(s, labels=["하", "중", "상"]):
    """값의 백분위(rank pct)에 따라 하/중/상 부여"""
    pct = s.rank (pct=True, method="average")
    return pd.cut(pct, bins=[0, 1/3, 2/3, 1], labels=labels, include_lowest=True)

likes = likes_raw.copy()
likes["likes_level"] = percentile_level(likes["likes"])
likes["views_level"] = percentile_level(likes["views"])

print("likes_level:", likes["likes_level"].value_counts())
print("views_level:", likes["views_level"].value_counts())


likes_level: likes_level
상    6633
중    6566
하    6539
Name: count, dtype: int64
views_level: views_level
상    6580
하    6579
중    6579
Name: count, dtype: int64


In [18]:
merged = (
    pairs
    .merge(likes,left_on ="item", right_on="items", how="left")
    .merge(title_raw, left_on="items", right_on="items", how="left"))

print(f"결합 결과: {merged.shape[0]:,} 행 x {merged.shape[1]:,} 열")
merged.head(5)

결합 결과: 719,405 행 x 10 열


,user,item,timestamp,datetime,items,likes,views,likes_level,views_level,title
0,36121,9580,1583378629552,2020-03-05 03:23:49.552,9580,231000,2567000,상,상,the best place to destroy the most wonderful team with the fastest speed
1,26572,9580,1583436719018,2020-03-05 19:31:59.018,9580,231000,2567000,상,상,the best place to destroy the most wonderful team with the fastest speed
2,94805,9580,1584083806481,2020-03-13 07:16:46.481,9580,231000,2567000,상,상,the best place to destroy the most wonderful team with the fastest speed
3,37550,9580,1584412681021,2020-03-17 02:38:01.021,9580,231000,2567000,상,상,the best place to destroy the most wonderful team with the fastest speed
4,89825,9580,1584649439020,2020-03-19 20:23:59.020,9580,231000,2567000,상,상,the best place to destroy the most wonderful team with the fastest speed


In [23]:
pairs_raw["datetime"] = pd.to_datetime(
    pairs_raw["timestamp"],
    unit="s"
)

In [19]:
stats = likes[["likes", "views"]].agg(["mean", "median", "min", "max", "var", "std"]).T
print(stats)

               mean    median      min         max           var           std
likes  6.649281e+04   31000.0  10000.0   4731000.0  1.533383e+10  1.238298e+05
views  2.000471e+06  977000.0  44000.0  93373000.0  1.218690e+13  3.490974e+06


In [24]:
pairs_raw["datetime"] = pd.to_datetime(pairs_raw["datetime"])
daily = pairs_raw.groupby(
    pairs_raw["datetime"].dt.date
).size()

daily.head()


ValueError: year must be in 1..9999, not 52145